# CreditWise AI - Model Persistence and Inference Pipeline

## Objective

In the previous notebook, multiple machine learning models were trained and evaluated for credit default prediction. The final selected solution was an ensemble of LightGBM and XGBoost models, which achieved the best overall performance.

This notebook focuses on preparing the solution for deployment by:

- Retraining the final models on the complete training dataset
- Saving trained model artifacts
- Saving feature schema information
- Building a reusable prediction pipeline
- Generating business-friendly risk scores
- Defining risk categories
- Testing inference on sample applicants
- Preparing assets for API and dashboard integration

## Expected Outputs

By the end of this notebook, the following deployment-ready artifacts will be available:

- LightGBM Model
- XGBoost Model
- Ensemble Prediction Pipeline
- Feature Schema
- Risk Scoring Logic
- Deployment Assets

These artifacts will later be integrated into a FastAPI backend, PostgreSQL database, explainability module, and interactive dashboard.

In [ ]:
import pandas as pd
import numpy as np

import joblib

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

## Step 1: Load Processed Dataset

The fully preprocessed dataset generated during feature engineering is loaded for final model training and deployment preparation.

In [ ]:
train_df = pd.read_csv(
    "../data/processed/train_processed.csv"
)

train_df.shape

## Step 2: Remove Non-Predictive Identifier Columns

Previous experiments showed that customer identifier columns do not contribute meaningful predictive information and should not be used during deployment.

In [ ]:
train_df = train_df.drop(
    columns=["SK_ID_CURR"],
    errors="ignore"
)

train_df.shape

## Step 3: Separate Features and Target

The dataset is separated into predictor variables and the target variable for model training.

In [ ]:
X = train_df.drop("TARGET", axis=1)
y = train_df["TARGET"]

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

## Step 4: Train Final Deployment Models

Model selection and evaluation have already been completed in the previous notebook.

For deployment, the final selected models are retrained using the complete dataset so they can learn from all available observations.

The following models will be trained:

- XGBoost
- LightGBM

These models will later be combined through probability averaging to form the final ensemble used in production.

In [ ]:
xgb_final = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
    random_state=42,
    eval_metric="logloss"
)

xgb_final.fit(X, y)

print("Final XGBoost Training Complete")

In [ ]:
lgbm_final = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    class_weight="balanced",
    random_state=42
)

lgbm_final.fit(X, y)

print("Final LightGBM Training Complete")